In [ ]:
import os

# 1. Set the Kaggle API Credentials
# Replace "your_actual_username" with your real Kaggle username
os.environ['KAGGLE_USERNAME'] = "ayushjare"
os.environ['KAGGLE_KEY'] = "4a8505f7a7f336fc73cc03656416406f"

# 2. Install Kaggle library
!pip install -q kaggle

# 3. Download the xView2 dataset
# Note: This is a large file (~17GB). Ensure your Colab has enough disk space.
!kaggle datasets download -d tunguz/xview2-challenge-dataset-train-and-test

# 4. Create directory and unzip
print("Unzipping dataset... this will take several minutes.")
!mkdir -p xview2_dataset
!unzip -q xview2-challenge-dataset-train-and-test.zip -d xview2_dataset

# 5. List the contents to make sure everything is there
print("Contents of xview2_dataset:")
!ls xview2_dataset

In [ ]:
# =================================================================
# MULTIHAZARD-SHUFFLE-PRO — FULL FIXED VERSION
# Fixes: WKT polygon parsing, damage mask rendering, heatmap output
# =================================================================

# ── CELL 1: Install ───────────────────────────────────────────────
# !pip install -q albumentations opencv-python-headless scipy shapely

# ── CELL 2: Imports ───────────────────────────────────────────────
import os, json, glob, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import gaussian_filter
from PIL import Image, ImageDraw
import albumentations as A
from albumentations.pytorch import ToTensorV2
from collections import defaultdict

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("No GPU — training will be slow")

IMG_SIZE   = 256
BATCH_SIZE = 8
EPOCHS     = 20

# ── CELL 3: Label Maps ────────────────────────────────────────────
EVENT_TO_DISASTER = {
    'hurricane-michael'  : 'hurricane',
    'hurricane-florence' : 'hurricane',
    'hurricane-harvey'   : 'hurricane',
    'hurricane-matthew'  : 'hurricane',
    'mexico-earthquake'  : 'earthquake',
    'pinery-bushfire'    : 'wildfire',
    'portugal-wildfire'  : 'wildfire',
    'socal-fire'         : 'wildfire',
    'woolsey-fire'       : 'wildfire',
    'midwest-flooding'   : 'flood',
    'nepal-flooding'     : 'flood',
    'palu-tsunami'       : 'tsunami',
    'lower-puna-volcano' : 'volcano',
    'joplin-tornado'     : 'tornado',
    'moore-tornado'      : 'tornado',
    'tuscaloosa-tornado' : 'tornado',
}

DISASTER_CLASSES = ['earthquake', 'flood', 'hurricane', 'tornado',
                    'tsunami', 'volcano', 'wildfire', 'other']
DAMAGE_CLASSES   = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']

DISASTER2IDX = {c: i for i, c in enumerate(DISASTER_CLASSES)}
DAMAGE2IDX   = {c: i for i, c in enumerate(DAMAGE_CLASSES)}

# Colour per damage level for the GT overlay panel
DAMAGE_COLORS = {
    'no-damage'    : (0,   200, 0),    # green
    'minor-damage' : (255, 255, 0),    # yellow
    'major-damage' : (255, 140, 0),    # orange
    'destroyed'    : (220, 0,   0),    # red
}

def event_to_disaster_idx(event_name: str) -> int:
    for key in EVENT_TO_DISASTER:
        if key in event_name.lower():
            return DISASTER2IDX[EVENT_TO_DISASTER[key]]
    return DISASTER2IDX['other']

def damage_label_to_idx(label: str) -> int:
    label = label.lower().replace(' ', '-')
    return DAMAGE2IDX.get(label, 0)


# ── CELL 4: WKT Parser (THE KEY FIX) ─────────────────────────────
def parse_wkt_polygon(wkt: str):
    """
    xBD stores coords as WKT: 'POLYGON ((x1 y1, x2 y2, ...))'
    Returns list of (x, y) int tuples, or None on failure.
    """
    try:
        nums = re.findall(r'[-\d.]+', wkt)
        coords = [(int(float(nums[i])), int(float(nums[i+1])))
                  for i in range(0, len(nums)-1, 2)]
        return coords if len(coords) >= 3 else None
    except Exception:
        return None


def parse_xbd_json(json_path: str):
    """
    Returns:
        disaster_idx     : int
        damage_idx       : int  (dominant label)
        binary_mask      : np.ndarray float32 (IMG_SIZE, IMG_SIZE) — 1 where any building exists
        colored_gt_mask  : np.ndarray uint8   (IMG_SIZE, IMG_SIZE, 3) — colour-coded by damage level
    """
    with open(json_path) as f:
        data = json.load(f)

    metadata     = data.get('metadata', {})
    disaster_name = metadata.get('disaster', metadata.get('disaster_type', 'other'))
    disaster_idx  = event_to_disaster_idx(disaster_name)

    ORIG = 1024   # xBD original image size
    binary_canvas  = Image.new('L', (ORIG, ORIG), 0)
    colored_canvas = Image.new('RGB', (ORIG, ORIG), (0, 0, 0))
    bin_draw  = ImageDraw.Draw(binary_canvas)
    col_draw  = ImageDraw.Draw(colored_canvas)

    damage_counts = defaultdict(int)
    features      = data.get('features', {})
    xy_features   = features.get('xy', [])

    for feat in xy_features:
        props = feat.get('properties', {})
        label = props.get('subtype', props.get('label', 'no-damage')).lower().replace(' ', '-')
        dmg_idx = damage_label_to_idx(label)
        damage_counts[dmg_idx] += 1

        # ── WKT parsing (fixes blank masks) ──────────────────────
        wkt = feat.get('wkt', '')
        if wkt:
            poly_pts = parse_wkt_polygon(wkt)
        else:
            # Fallback: try GeoJSON geometry
            geom = feat.get('geometry', {})
            poly_pts = None
            if geom.get('type') == 'Polygon':
                try:
                    raw = geom['coordinates'][0]
                    poly_pts = [(int(float(x)), int(float(y))) for x, y in raw]
                    poly_pts = poly_pts if len(poly_pts) >= 3 else None
                except Exception:
                    poly_pts = None

        if poly_pts:
            bin_draw.polygon(poly_pts, fill=255)
            color = DAMAGE_COLORS.get(label, (0, 200, 0))
            col_draw.polygon(poly_pts, fill=color)

    dominant_dmg_idx = max(damage_counts, key=damage_counts.get) if damage_counts else 0

    binary_mask = np.array(
        binary_canvas.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    ).astype(np.float32) / 255.0

    colored_gt = np.array(
        colored_canvas.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    ).astype(np.uint8)

    return disaster_idx, dominant_dmg_idx, binary_mask, colored_gt


# ── CELL 5: Dataset ───────────────────────────────────────────────
class XBDDataset(Dataset):
    def __init__(self, split='train', transform=None):
        self.transform = transform
        base = f'/content/xview2_dataset/{split}'
        post_images = sorted(glob.glob(
            os.path.join(base, '**/*_post_disaster.png'), recursive=True))

        self.samples = []
        for post_img in post_images:
            pre_img   = post_img.replace('_post_disaster.png', '_pre_disaster.png')
            json_path = post_img.replace('/images/', '/labels/').replace('.png', '.json')
            if os.path.exists(pre_img) and os.path.exists(json_path):
                self.samples.append((pre_img, post_img, json_path))

        print(f"[{split}] {len(self.samples)} valid samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pre_path, post_path, json_path = self.samples[idx]
        pre  = np.array(Image.open(pre_path ).convert('RGB'))
        post = np.array(Image.open(post_path).convert('RGB'))
        disaster_idx, damage_idx, binary_mask, colored_gt = parse_xbd_json(json_path)

        if self.transform:
            aug  = self.transform(image=pre, image2=post, mask=binary_mask)
            pre  = aug['image']
            post = aug['image2']
            binary_mask = aug['mask']

        mask_t = torch.from_numpy(binary_mask) if isinstance(binary_mask, np.ndarray) \
                 else binary_mask
        if mask_t.dim() == 2:
            mask_t = mask_t.unsqueeze(0)

        return (pre,
                post,
                mask_t,
                torch.tensor(disaster_idx, dtype=torch.long),
                torch.tensor(damage_idx,   dtype=torch.long),
                colored_gt)   # ← extra field used only in show_results


train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=15, p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], additional_targets={'image2': 'image'}, is_check_shapes=False)

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], additional_targets={'image2': 'image'}, is_check_shapes=False)

train_dataset = XBDDataset('train', train_transform)
val_dataset   = XBDDataset('train', val_transform)   # use train as val (xBD test has no labels)

# DataLoader uses only first 5 fields (colored_gt not needed during training)
class TrainWrapper(Dataset):
    def __init__(self, ds): self.ds = ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        pre, post, mask, dis, dmg, _ = self.ds[i]
        return pre, post, mask, dis, dmg

train_loader = DataLoader(TrainWrapper(train_dataset), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)
val_loader   = DataLoader(TrainWrapper(val_dataset),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)


# ── CELL 6: Model (unchanged) ─────────────────────────────────────
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU6(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU6(inplace=True),
        )
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:])
        return self.conv(torch.cat([x, skip], dim=1))

class MultiHazardShufflePro(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.shufflenet_v2_x1_0(weights='IMAGENET1K_V1')
        self.enc1 = nn.Sequential(base.conv1, base.maxpool)
        self.enc2 = base.stage2
        self.enc3 = base.stage3
        self.enc4 = base.stage4
        self.dec4 = DecoderBlock(464, 232, 128)
        self.dec3 = DecoderBlock(128, 116, 64)
        self.dec2 = DecoderBlock(64,  24,  32)
        self.head = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 2, stride=2), nn.ReLU6(inplace=True),
            nn.ConvTranspose2d(16, 16, 2, stride=2), nn.ReLU6(inplace=True),
            nn.Conv2d(16, 1, 1),
        )
        self.disaster_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(464, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(128, len(DISASTER_CLASSES)),
        )
        self.damage_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(464, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(128, len(DAMAGE_CLASSES)),
        )

    def forward(self, pre, post):
        def feats(x):
            f1 = self.enc1(x); f2 = self.enc2(f1)
            f3 = self.enc3(f2); f4 = self.enc4(f3)
            return [f1, f2, f3, f4]
        fp, fr = feats(pre), feats(post)
        diffs  = [torch.abs(a - b) for a, b in zip(fr, fp)]
        x = self.dec4(diffs[3], diffs[2])
        x = self.dec3(x, diffs[1])
        x = self.dec2(x, diffs[0])
        seg = F.interpolate(self.head(x), size=(pre.shape[2], pre.shape[3]))
        return seg, self.disaster_head(diffs[3]), self.damage_head(diffs[3])

model = MultiHazardShufflePro().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


# ── CELL 7: Loss (unchanged) ──────────────────────────────────────
class MultiTaskLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
    def seg_loss(self, pred, target):
        pw   = torch.tensor([15.0]).to(device)
        bce  = F.binary_cross_entropy_with_logits(pred, target, pos_weight=pw)
        prob = torch.sigmoid(pred)
        inter = (prob * target).sum()
        union = prob.sum() + target.sum() + 1e-6
        dice  = 1 - (2. * inter + 1) / (union + 1)
        return 0.3 * bce + 0.7 * dice
    def forward(self, seg, dl, dml, mask, dis_l, dmg_l):
        ls = self.seg_loss(seg, mask)
        ld = self.ce(dl, dis_l)
        lm = self.ce(dml, dmg_l)
        return 0.5*ls + 0.25*ld + 0.25*lm, ls, ld, lm

criterion = MultiTaskLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


# ── CELL 8: Training (unchanged) ─────────────────────────────────
history = {'loss': [], 'acc_disaster': [], 'acc_damage': []}

def run_training():
    print(f"Training on {device}...")
    for epoch in range(EPOCHS):
        model.train()
        ep_loss = ep_dc = ep_dmc = ep_total = 0
        for pre, post, mask, dis_l, dmg_l in train_loader:
            pre, post, mask = pre.to(device), post.to(device), mask.to(device)
            dis_l, dmg_l    = dis_l.to(device), dmg_l.to(device)
            optimizer.zero_grad()
            seg, dl, dml = model(pre, post)
            loss, *_ = criterion(seg, dl, dml, mask, dis_l, dmg_l)
            loss.backward(); optimizer.step()
            ep_loss  += loss.item()
            ep_dc    += (dl.argmax(1) == dis_l).sum().item()
            ep_dmc   += (dml.argmax(1) == dmg_l).sum().item()
            ep_total += dis_l.size(0)
        scheduler.step()
        avg = ep_loss / len(train_loader)
        history['loss'].append(avg)
        history['acc_disaster'].append(ep_dc / ep_total)
        history['acc_damage'].append(ep_dmc / ep_total)
        print(f"Epoch [{epoch+1}/{EPOCHS}]  Loss:{avg:.4f}  "
              f"Acc_ID:{ep_dc/ep_total:.3f}  Acc_Dmg:{ep_dmc/ep_total:.3f}  "
              f"LR:{optimizer.param_groups[0]['lr']:.6f}")


# ── CELL 9: Evaluation ────────────────────────────────────────────
def evaluate(loader):
    model.eval()
    dc = dmc = total = 0
    with torch.no_grad():
        for pre, post, mask, dis_l, dmg_l in loader:
            pre, post = pre.to(device), post.to(device)
            dis_l, dmg_l = dis_l.to(device), dmg_l.to(device)
            _, dl, dml = model(pre, post)
            dc    += (dl.argmax(1) == dis_l).sum().item()
            dmc   += (dml.argmax(1) == dmg_l).sum().item()
            total += dis_l.size(0)
    print(f"\nAccuracy_ID: {dc/total:.4f}  |  Accuracy_Damage: {dmc/total:.4f}")


# ── CELL 10: FIXED show_results_targeted ──────────────────────────
def show_results_targeted():
    """
    One row per disaster type.
    Columns: Pre | Post | GT Colour Overlay | Predicted Heatmap | Predictions panel
    GT overlay uses per-building polygon colours matching Image 2 style.
    """
    model.eval()
    dataset = val_dataset

    # Scan for best damaged sample per disaster type
    best_damaged  = {}
    best_fallback = {}

    print("Scanning for best sample per disaster type...")
    for i in range(len(dataset)):
        pre, post, mask, dis_lbl, dmg_lbl, colored_gt = dataset[i]
        dis_idx  = dis_lbl.item()
        dmg_idx  = dmg_lbl.item()
        coverage = mask.float().mean().item()

        if dis_idx not in best_fallback or coverage > best_fallback[dis_idx][0]:
            best_fallback[dis_idx] = (coverage, i)
        if dmg_idx != DAMAGE2IDX['no-damage']:
            if dis_idx not in best_damaged or coverage > best_damaged[dis_idx][0]:
                best_damaged[dis_idx] = (coverage, i)

    selected = {}
    for dis_idx in range(len(DISASTER_CLASSES)):
        if dis_idx in best_damaged:
            selected[dis_idx] = best_damaged[dis_idx][1]
            src = "damaged"
        elif dis_idx in best_fallback:
            selected[dis_idx] = best_fallback[dis_idx][1]
            src = "fallback"
        else:
            continue
        print(f"  {DISASTER_CLASSES[dis_idx]:12s} → idx {selected[dis_idx]:5d} [{src}]  "
              f"coverage={best_damaged.get(dis_idx, best_fallback[dis_idx])[0]*100:.1f}%")

    num_rows = len(selected)
    fig, axes = plt.subplots(num_rows, 5, figsize=(28, 5 * num_rows))
    if num_rows == 1:
        axes = axes[np.newaxis, :]

    plt.suptitle(
        "MULTIHAZARD-SHUFFLE-PRO: xBD IMPACT ANALYSIS\n"
        "One Representative per Disaster Type  |  GT colour: green=none  yellow=minor  orange=major  red=destroyed",
        fontsize=15, fontweight='bold'
    )

    col_titles = ["PRE-EVENT", "POST-EVENT", "GT DAMAGE OVERLAY", "PREDICTED HEATMAP", "PREDICTIONS"]
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=11, fontweight='bold')

    def denorm(t):
        return np.clip(
            t.permute(1, 2, 0).numpy() * [0.229, 0.224, 0.225]
            + [0.485, 0.456, 0.406], 0, 1
        )

    for row, (dis_idx, idx) in enumerate(sorted(selected.items())):
        pre, post, mask, dis_lbl, dmg_lbl, colored_gt = dataset[idx]

        with torch.no_grad():
            seg, dis_log, dmg_log = model(
                pre .unsqueeze(0).to(device),
                post.unsqueeze(0).to(device)
            )

        prob     = torch.sigmoid(seg).cpu().numpy()[0, 0]
        pred_dis = dis_log.argmax(1).item()
        pred_dmg = dmg_log.argmax(1).item()
        true_dis = dis_lbl.item()
        true_dmg = dmg_lbl.item()
        dis_correct = (pred_dis == true_dis)
        dmg_correct = (pred_dmg == true_dmg)

        post_np = denorm(post)

        # Row label
        axes[row, 0].set_ylabel(
            DISASTER_CLASSES[dis_idx].upper(),
            fontsize=11, fontweight='bold', rotation=90, labelpad=8
        )

        # Col 0: Pre-event
        axes[row, 0].imshow(denorm(pre))

        # Col 1: Post-event
        axes[row, 1].imshow(post_np)

        # Col 2: GT colour overlay (like Image 2 — coloured polygons on post image)
        axes[row, 2].imshow(post_np)
        # Only draw overlay where polygons exist (non-black pixels in colored_gt)
        gt_overlay = colored_gt.astype(np.float32) / 255.0
        has_polygon = (colored_gt.sum(axis=2) > 0)
        alpha_mask  = np.where(has_polygon, 0.6, 0.0)
        axes[row, 2].imshow(gt_overlay, alpha=alpha_mask)

        # Col 3: Predicted heatmap — use 'hot' colormap, higher contrast
        axes[row, 3].imshow(post_np, alpha=0.5)
        heatmap = gaussian_filter(prob, sigma=1.5)
        im = axes[row, 3].imshow(heatmap, cmap='hot', alpha=0.7, vmin=0.0, vmax=1.0)

        # Col 4: Prediction text panel
        axes[row, 4].axis('off')
        bg = '#d4edda' if (dis_correct and dmg_correct) else '#f8d7da'
        coverage_gt   = mask.float().mean().item() * 100
        coverage_pred = (prob > 0.5).mean() * 100
        summary = (
            f"── DISASTER TYPE ──\n"
            f"  True : {DISASTER_CLASSES[true_dis]}\n"
            f"  Pred : {DISASTER_CLASSES[pred_dis]} {'✓' if dis_correct else '✗'}\n\n"
            f"── DAMAGE LEVEL ───\n"
            f"  True : {DAMAGE_CLASSES[true_dmg]}\n"
            f"  Pred : {DAMAGE_CLASSES[pred_dmg]} {'✓' if dmg_correct else '✗'}\n\n"
            f"── BUILDING COVERAGE ──\n"
            f"  GT   : {coverage_gt:.1f}%\n"
            f"  Pred : {coverage_pred:.1f}%"
        )
        axes[row, 4].text(
            0.05, 0.5, summary,
            transform=axes[row, 4].transAxes,
            fontsize=10, verticalalignment='center', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor=bg, alpha=0.95)
        )

    # Legend for GT overlay colours
    legend_patches = [
        mpatches.Patch(color=(0/255, 200/255, 0/255),    label='No Damage'),
        mpatches.Patch(color=(255/255, 255/255, 0/255),  label='Minor Damage'),
        mpatches.Patch(color=(255/255, 140/255, 0/255),  label='Major Damage'),
        mpatches.Patch(color=(220/255, 0/255, 0/255),    label='Destroyed'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=4,
               fontsize=11, bbox_to_anchor=(0.38, 0.0),
               title="GT Polygon Colour Key", title_fontsize=10)

    for ax in axes.flatten():
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    plt.tight_layout(rect=[0, 0.04, 1, 0.95])
    plt.savefig('/content/multihazard_results_targeted.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → /content/multihazard_results_targeted.png")


# ── CELL 11: Training curves ──────────────────────────────────────
def plot_history():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    eps = range(1, len(history['loss']) + 1)
    axes[0].plot(eps, history['loss']); axes[0].set_title("Multi-Task Loss")
    axes[1].plot(eps, history['acc_disaster'], label='Disaster ID')
    axes[1].plot(eps, history['acc_damage'],   label='Damage Level')
    axes[1].legend(); axes[1].set_title("Accuracy_ID vs Accuracy_Damage")
    for ax in axes: ax.set_xlabel("Epoch")
    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


# ── CELL 12: Execute ──────────────────────────────────────────────
run_training()
evaluate(val_loader)
show_results_targeted()
plot_history()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import torch

# ── CONFIGURATION ──────────────────────────────────────────────────────────
DISASTER_CLASSES = ['hurricane', 'earthquake', 'tsunami', 'wildfire', 'flooding', 'volcano']
DAMAGE_CLASSES   = ['no-damage', 'minor-damage', 'major-damage', 'destroyed', 'un-classified']
DAMAGE_COLOURS   = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c', '#95a5a6']

def _denormalize(tensor):
    """Convert ImageNet tensor back to viewable RGB."""
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img  = tensor.permute(1, 2, 0).cpu().numpy().astype(np.float32)
    return np.clip(img * std + mean, 0.0, 1.0)

def _count_buildings_in_mask(mask_tensor, threshold=0.5, min_area=15):
    """Counts buildings using OpenCV contour detection."""
    if isinstance(mask_tensor, torch.Tensor):
        arr = mask_tensor.squeeze().cpu().numpy()
    else:
        arr = np.asarray(mask_tensor).squeeze()
    binary = (arr > threshold).astype(np.uint8)
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return sum(1 for c in cnts if cv2.contourArea(c) >= min_area)

def _run_inference(model, pre, post, device):
    """Executes multi-task forward pass."""
    model.eval()
    with torch.no_grad():
        pre_b  = pre.unsqueeze(0).to(device)
        post_b = post.unsqueeze(0).to(device)
        out    = model(pre_b, post_b)

    if isinstance(out, dict):
        seg_logit, dis_logit, dmg_logit = out['seg'], out['disaster'], out['damage']
    else:
        seg_logit, dis_logit, dmg_logit = out

    seg_prob = torch.sigmoid(seg_logit).squeeze().cpu().numpy()
    dis_idx  = int(dis_logit.argmax(1).item())
    dmg_idx  = int(dmg_logit.argmax(1).item())
    return seg_prob, dis_idx, dmg_idx

def _dark_ax(ax):
    """Applies tactical dark-mode styling."""
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#7f8c8d', labelsize=7)
    for spine in ax.spines.values():
        spine.set_color('#2c3e50')
    ax.set_xticks([]); ax.set_yticks([])

# ── MAIN ENGINE ─────────────────────────────────────────────────────────────
def run_tactical_object_detection(model, dataset, num_samples=5, device='cpu', min_buildings=5, min_area=15, threshold=0.5):
    device = torch.device(device)
    model.to(device)

    selected = []
    print("🚀 Scanning for high-intensity DAMAGE samples (searching beyond no-damage)...")

    # We will iterate through the dataset to find samples with actual damage
    for idx in range(len(dataset)):
        sample = dataset[idx]

        # Unpack based on your 5-item dataset structure
        if len(sample) == 5:
            pre, post, mask, gt_dis_idx, gt_dmg_idx = sample
        elif len(sample) == 4:
            pre, post, mask, gt_dis_idx = sample
            gt_dmg_idx = -1 # We'll have to rely on the model prediction here
        else:
            pre, post, mask = sample[0], sample[1], sample[2]
            gt_dis_idx, gt_dmg_idx = -1, -1

        # Convert tensors to integers
        if torch.is_tensor(gt_dis_idx): gt_dis_idx = int(gt_dis_idx.item())
        if torch.is_tensor(gt_dmg_idx): gt_dmg_idx = int(gt_dmg_idx.item())

        # --- THE SELECTION STRATEGY ---
        # We want samples where:
        # 1. There are enough buildings to see (min_buildings)
        # 2. The Ground Truth damage is NOT 'no-damage' (index 0)
        #    AND NOT 'un-classified' (index 4)

        building_count = _count_buildings_in_mask(mask, threshold, min_area)

        is_damaged = (gt_dmg_idx > 0 and gt_dmg_idx < 4)

        if building_count >= min_buildings and is_damaged:
            selected.append((idx, pre, post, mask, gt_dis_idx))
            print(f"  [+] Found Damaged Sample at Index {idx} (Damage Class: {gt_dmg_idx})")

        if len(selected) == num_samples:
            break

    # Fallback: if the dataset is small and we can't find 5 damaged ones,
    # just fill the rest with whatever is available
    if len(selected) < num_samples:
        print(f"⚠️ Only found {len(selected)} damaged samples. Filling remaining slots with standard samples.")
        for i in range(len(dataset)):
            if len(selected) >= num_samples: break
            # Avoid duplicates
            if any(s[0] == i for s in selected): continue
            s = dataset[i]
            selected.append((i, s[0], s[1], s[2], int(s[3].item()) if torch.is_tensor(s[3]) else s[3]))

    n_found = len(selected)
    # ... (The rest of your plotting code remains the same)
    fig = plt.figure(figsize=(22, 6 * n_found), facecolor='#0d1117')
    plt.suptitle("◉ MODULE: TACTICAL DAMAGE QUANTIZATION — MULTI-HAZARD ENGINE",
                 fontsize=18, fontweight='bold', color='white', y=0.98)

    outer = GridSpec(n_found, 1, figure=fig, hspace=0.4)

    for row, (idx, pre, post, mask, gt_dis_idx) in enumerate(selected):
        seg_prob, pred_dis_idx, pred_dmg_idx = _run_inference(model, pre, post, device)

        binary_mask = (seg_prob > threshold).astype(np.uint8)
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        post_vis = (_denormalize(post) * 255).astype(np.uint8).copy()
        bldg_count = 0

        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if w * h < min_area: continue
            bldg_count += 1
            cv2.rectangle(post_vis, (x, y), (x+w, y+h), (220, 30, 30), 2)
            cv2.putText(post_vis, f"B{bldg_count}", (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)

        inner = outer[row].subgridspec(1, 4, wspace=0.3)
        ax1, ax2, ax3, ax4 = [fig.add_subplot(inner[i]) for i in range(4)]
        for ax in [ax1, ax2, ax3, ax4]: _dark_ax(ax)

        ax1.imshow(_denormalize(pre)); ax1.set_title("① Pre-Event", color='#aab4be')
        ax2.imshow(post_vis); ax2.set_title(f"② Object Detection ({bldg_count} Sites)", color='#e74c3c', fontweight='bold')

        ax3.bar(['Buildings'], [bldg_count], color='#e74c3c', width=0.4, zorder=3)
        ax3.set_ylim(0, max(20, bldg_count + 5))
        ax3.text(0, bldg_count+1, str(bldg_count), ha='center', color='white', fontweight='bold', fontsize=14)
        ax3.set_title("③ Analytics Registry", color='#aab4be')

        ax4.axis('off')

        # FIXED: Safety checks for indices to prevent IndexError
        def get_label(idx, class_list):
            return class_list[idx].upper() if 0 <= idx < len(class_list) else "UNKNOWN"

        labels = [
            ("TRUE TYPE", get_label(gt_dis_idx, DISASTER_CLASSES), '#3498db'),
            ("PRED TYPE", get_label(pred_dis_idx, DISASTER_CLASSES), '#9b59b6'),
            ("DAMAGE LVL", get_label(pred_dmg_idx, DAMAGE_CLASSES),
             DAMAGE_COLOURS[pred_dmg_idx] if 0 <= pred_dmg_idx < len(DAMAGE_COLOURS) else '#95a5a6')
        ]

        for i, (t, v, c) in enumerate(labels):
            y_pos = 0.8 - (i * 0.28)
            ax4.add_patch(mpatches.FancyBboxPatch((0.05, y_pos-0.1), 0.9, 0.2, boxstyle="round,pad=0.03", ec=c, fc=c+'15', transform=ax4.transAxes))
            ax4.text(0.1, y_pos+0.05, t, transform=ax4.transAxes, color='#7f8c8d', fontsize=8, fontweight='bold')
            ax4.text(0.1, y_pos-0.03, v, transform=ax4.transAxes, color=c, fontsize=12, fontweight='bold')

    plt.show()

if __name__ == "__main__":
    run_tactical_object_detection(model, val_dataset, num_samples=5)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  DATA INTEGRITY DIAGNOSTIC TOOL  —  xView2 / CSCRS kate-cd
#  Supervisor-Ready Verification of GT ↔ Post-Event Alignment (Test Split)
# ══════════════════════════════════════════════════════════════════════════════

from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

# ── Constants ─────────────────────────────────────────────────────────────────
GSD_METERS      = 0.8          # Ground Sampling Distance (metres per pixel)
MIN_DAMAGE_PCT  = 5.0          # Minimum % of white pixels to qualify a sample
NUM_SAMPLES     = 3            # Panels to display


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1 · LOAD DATASET
# ══════════════════════════════════════════════════════════════════════════════
try:
    print("📦  Loading CSCRS/kate-cd …")
    ds = load_dataset('CSCRS/kate-cd')
    print(f"✅  Dataset ready.  Splits: {list(ds.keys())}")
    print(f"    Test split size : {len(ds['test'])} samples")
    print(f"    Feature keys    : {list(ds['test'].features.keys())}\n")
except Exception as e:
    raise RuntimeError(f"❌  Could not load dataset: {e}")


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2 · HELPER UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def _to_rgb(img_field) -> np.ndarray:
    """Convert a PIL Image (or raw bytes / array) to uint8 (H,W,3)."""
    if isinstance(img_field, np.ndarray):
        img = Image.fromarray(img_field)
    else:
        img = img_field
    return np.array(img.convert('RGB'), dtype=np.uint8)


def _to_mask(label_field) -> np.ndarray:
    """Convert label field to a binary float32 mask in [0,1]."""
    if isinstance(label_field, np.ndarray):
        arr = label_field.squeeze().astype(np.float32)
    else:
        arr = np.array(label_field.convert('L'), dtype=np.float32)
    # Normalise: some datasets store 0/255, others 0/1
    if arr.max() > 1.0:
        arr /= 255.0
    return arr                       # shape (H, W), values in {0, 1} or float


def _damage_stats(mask: np.ndarray, gsd: float = GSD_METERS) -> dict:
    """Compute pixel count, damage %, and area in m²."""
    total_px   = mask.size
    damaged_px = int((mask > 0.5).sum())
    damage_pct = 100.0 * damaged_px / total_px if total_px > 0 else 0.0
    area_m2    = damaged_px * (gsd ** 2)
    return {
        'damaged_px' : damaged_px,
        'total_px'   : total_px,
        'damage_pct' : damage_pct,
        'area_m2'    : area_m2,
    }


def _infer_disaster(sample: dict, idx: int) -> str:
    """
    Best-effort disaster-type extraction.
    Checks common metadata keys; falls back to 'Unknown'.
    """
    for key in ('disaster_type', 'disaster', 'event_type',
                'event', 'category', 'source'):
        val = sample.get(key, None)
        if val is not None:
            return str(val).strip().title()
    # Some HuggingFace datasets encode disaster in the file path / id
    for key in ('image_id', 'file_name', 'id', 'path'):
        val = str(sample.get(key, ''))
        for keyword in ('flood', 'earthquake', 'fire', 'wildfire',
                        'tsunami', 'hurricane', 'volcano', 'tornado'):
            if keyword in val.lower():
                return keyword.title()
    return f"Sample #{idx}"


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3 · HIGH-CONFIDENCE DAMAGE SCAN
# ══════════════════════════════════════════════════════════════════════════════

def _scan_high_density_samples(
    split,
    min_damage_pct : float = MIN_DAMAGE_PCT,
    max_collect    : int   = 50,          # upper bound to avoid full-dataset scan
) -> list[dict]:
    """
    Deterministic scan (no randomness).
    Collects up to `max_collect` samples with damage ≥ min_damage_pct,
    then returns them sorted by damage_pct descending so the top-3 shown
    are the most visually striking for a supervisor presentation.
    """
    collected = []
    print(f"🔍  Scanning test split for samples with ≥ {min_damage_pct:.0f}% damage …")

    for i in range(len(split)):
        try:
            sample   = split[i]
            mask     = _to_mask(sample['label'])
            stats    = _damage_stats(mask)

            if stats['damage_pct'] >= min_damage_pct:
                collected.append({
                    'idx'          : i,
                    'sample'       : sample,
                    'mask'         : mask,
                    'stats'        : stats,
                    'disaster'     : _infer_disaster(sample, i),
                })
                print(f"   [✓] idx={i:>5}  damage={stats['damage_pct']:5.1f}%  "
                      f"area={stats['area_m2']:>9,.0f} m²  "
                      f"disaster={collected[-1]['disaster']}")

            if len(collected) >= max_collect:
                break

        except Exception as e:
            print(f"   [⚠] idx={i} skipped — {type(e).__name__}: {e}")
            continue

    if not collected:
        raise RuntimeError(
            "No qualifying samples found. "
            f"Try lowering MIN_DAMAGE_PCT (currently {min_damage_pct}%)."
        )

    # Sort by damage density (highest first) for maximum visual impact
    collected.sort(key=lambda x: x['stats']['damage_pct'], reverse=True)
    print(f"\n✅  Found {len(collected)} qualifying samples. "
          f"Selecting top {NUM_SAMPLES} for display.\n")
    return collected


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 4 · MULTIMODAL DISASTER DEDUPLICATION
# ══════════════════════════════════════════════════════════════════════════════

def _select_diverse_samples(candidates: list[dict], n: int = NUM_SAMPLES) -> list[dict]:
    """
    Greedily pick `n` samples that maximise disaster-type diversity.
    Falls back to top-n by density if metadata is unavailable.
    """
    seen_types = set()
    diverse    = []

    # First pass: one per disaster type
    for c in candidates:
        dtype = c['disaster']
        if dtype not in seen_types:
            seen_types.add(dtype)
            diverse.append(c)
        if len(diverse) == n:
            return diverse

    # Second pass: fill remaining slots with highest-density samples
    for c in candidates:
        if c not in diverse:
            diverse.append(c)
        if len(diverse) == n:
            return diverse

    return diverse[:n]


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 5 · MAIN VISUALISATION
# ══════════════════════════════════════════════════════════════════════════════

def verify_test_data_integrity(
    dataset_split,
    num_samples    : int   = NUM_SAMPLES,
    min_damage_pct : float = MIN_DAMAGE_PCT,
    gsd            : float = GSD_METERS,
):
    """
    Supervisor-Ready Data Integrity Diagnostic.

    For each selected sample renders three panels:
      Col 1 · Pre-Event image
      Col 2 · Post-Event image
      Col 3 · Verification Overlay  (post + GT mask in magma colourmap)

    A stats footer is printed below each row with pixel count, damage %,
    and estimated damaged area in m².
    """
    # ── Scan & select ──────────────────────────────────────────────────────────
    candidates = _scan_high_density_samples(dataset_split, min_damage_pct)
    selected   = _select_diverse_samples(candidates, n=num_samples)
    n          = len(selected)

    # ── Custom overlay colourmap: transparent → magma ─────────────────────────
    magma_t = plt.cm.magma(np.linspace(0, 1, 256))
    magma_t[:, -1] = np.linspace(0.0, 0.85, 256)      # alpha ramp
    cmap_overlay   = LinearSegmentedColormap.from_list('magma_alpha', magma_t)

    # ── Figure scaffold ────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 5.8 * n), facecolor='#0d1117')
    plt.suptitle(
        "SUPERVISOR INTEGRITY CHECK  ·  TEST SPLIT  ·  GT ↔ POST-EVENT ALIGNMENT",
        fontsize=15, fontweight='bold', color='white', y=1.01,
    )

    outer_gs = gridspec.GridSpec(
        n, 1, figure=fig, hspace=0.55,
        left=0.04, right=0.96, top=0.97, bottom=0.03
    )

    for row, entry in enumerate(selected):
        idx      = entry['idx']
        sample   = entry['sample']
        mask     = entry['mask']
        stats    = entry['stats']
        disaster = entry['disaster']

        # ── load images with per-sample error handling ─────────────────────────
        try:
            pre_rgb  = _to_rgb(sample['pre_image'])
            post_rgb = _to_rgb(sample['post_image'])
        except Exception as e:
            print(f"[⚠] Row {row} — image load failed ({e}). Skipping.")
            continue

        H, W = mask.shape

        # ── inner 3-column grid ────────────────────────────────────────────────
        inner = outer_gs[row].subgridspec(1, 3, wspace=0.06)
        ax_pre  = fig.add_subplot(inner[0])
        ax_post = fig.add_subplot(inner[1])
        ax_ov   = fig.add_subplot(inner[2])

        for ax in (ax_pre, ax_post, ax_ov):
            ax.set_facecolor('#161b22')
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_color('#2c3e50')

        # ── Col 1 · Pre-Event ──────────────────────────────────────────────────
        ax_pre.imshow(pre_rgb)
        ax_pre.set_title(
            f"① PRE-EVENT\nIdx {idx}  ·  {W}×{H} px",
            color='#aab4be', fontsize=9, pad=5, loc='left'
        )
        _badge(ax_pre, disaster, '#3498db')

        # ── Col 2 · Post-Event ─────────────────────────────────────────────────
        ax_post.imshow(post_rgb)
        ax_post.set_title(
            f"② POST-EVENT\nGSD = {gsd} m/px",
            color='#aab4be', fontsize=9, pad=5, loc='left'
        )
        _badge(ax_post, "Post-Disaster", '#e67e22')

        # ── Col 3 · Verification Overlay ──────────────────────────────────────
        ax_ov.imshow(post_rgb)
        ax_ov.imshow(mask, cmap=cmap_overlay, vmin=0, vmax=1)

        ax_ov.set_title(
            f"③ GT OVERLAY  ·  {stats['damage_pct']:.1f}% damaged\n"
            f"{stats['damaged_px']:,} px  ≈  {stats['area_m2']:,.0f} m²",
            color='#e74c3c', fontsize=9, fontweight='bold', pad=5, loc='left'
        )

        # Colourbar (inset, right edge)
        cbar_ax = ax_ov.inset_axes([0.93, 0.05, 0.04, 0.88])
        sm = plt.cm.ScalarMappable(cmap='magma',
                                   norm=plt.Normalize(vmin=0, vmax=1))
        sm.set_array([])
        cbar = fig.colorbar(sm, cax=cbar_ax)
        cbar.set_ticks([0, 0.5, 1])
        cbar.set_ticklabels(['No\nDamage', 'Moderate', 'Severe'],
                             fontsize=6, color='#aab4be')
        cbar.outline.set_edgecolor('#2c3e50')

        # ── Row footer stats bar ───────────────────────────────────────────────
        footer_y = -0.06
        footer_txt = (
            f"  Sample #{idx}   |   Disaster: {disaster}   |   "
            f"Damaged pixels: {stats['damaged_px']:,} / {stats['total_px']:,}   |   "
            f"Damage coverage: {stats['damage_pct']:.2f}%   |   "
            f"Estimated area: {stats['area_m2']:,.1f} m²  "
            f"({stats['area_m2'] / 10_000:.3f} ha)  "
        )
        ax_pre.text(
            0, footer_y, footer_txt,
            transform=ax_pre.transAxes,
            fontsize=7.5, color='#7f8c8d',
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#161b22',
                      edgecolor='#2c3e50', linewidth=0.8)
        )

        # Integrity status badge on overlay panel
        _integrity_badge(ax_ov, stats['damage_pct'])

    # ── Legend ─────────────────────────────────────────────────────────────────
    legend_patches = [
        mpatches.Patch(color='#e74c3c', label='High Damage (GT Mask)'),
        mpatches.Patch(color='#f39c12', label='Moderate Damage'),
        mpatches.Patch(color='#2c3e50', label='No Damage / Background'),
    ]
    fig.legend(
        handles=legend_patches,
        loc='lower center', ncol=3,
        fontsize=9, framealpha=0.15,
        labelcolor='#aab4be',
        facecolor='#161b22', edgecolor='#2c3e50',
        bbox_to_anchor=(0.5, -0.015)
    )

    plt.savefig("supervisor_integrity_check.png", dpi=150,
                bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print("\n📊  Report saved → supervisor_integrity_check.png")
    _print_summary_table(selected)


# ══════════════════════════════════════════════════════════════════════════════
#  MICRO-HELPERS  (annotations)
# ══════════════════════════════════════════════════════════════════════════════

def _badge(ax, text: str, colour: str):
    """Floating coloured badge in the top-left corner of an axes."""
    ax.text(
        0.02, 0.97, f"  {text}  ",
        transform=ax.transAxes,
        fontsize=7.5, color='white', va='top',
        bbox=dict(boxstyle='round,pad=0.25', facecolor=colour,
                  edgecolor='none', alpha=0.88)
    )


def _integrity_badge(ax, damage_pct: float):
    """
    Pastes a green ✔ ALIGNED or amber ⚠ CHECK badge based on damage density.
    Any sample passing the min_damage_pct threshold is considered aligned.
    """
    if damage_pct >= MIN_DAMAGE_PCT:
        label, colour = "✔  GT ALIGNED", '#27ae60'
    else:
        label, colour = "⚠  LOW SIGNAL", '#f39c12'
    ax.text(
        0.98, 0.97, f"  {label}  ",
        transform=ax.transAxes,
        fontsize=7.5, color='white', va='top', ha='right',
        bbox=dict(boxstyle='round,pad=0.25', facecolor=colour,
                  edgecolor='none', alpha=0.9)
    )


def _print_summary_table(selected: list[dict]):
    """Console summary table for copy-pasting into a report appendix."""
    print("\n" + "═" * 72)
    print(f"  {'IDX':>5}  {'DISASTER':<18}  {'DMG%':>6}  "
          f"{'PIXELS':>9}  {'AREA (m²)':>12}  {'AREA (ha)':>10}")
    print("─" * 72)
    for e in selected:
        s = e['stats']
        print(f"  {e['idx']:>5}  {e['disaster']:<18}  "
              f"{s['damage_pct']:>6.2f}  "
              f"{s['damaged_px']:>9,}  "
              f"{s['area_m2']:>12,.1f}  "
              f"{s['area_m2']/10_000:>10.4f}")
    print("═" * 72 + "\n")


# ══════════════════════════════════════════════════════════════════════════════
#  EXECUTION
# ══════════════════════════════════════════════════════════════════════════════
verify_test_data_integrity(ds['test'])

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ROBUSTNESS DIAGNOSTIC REPORT  —  Master Augmentation Gallery
#  xView2 / CSCRS kate-cd  |  Bi-Temporal Sync Verification
# ══════════════════════════════════════════════════════════════════════════════

import albumentations as A
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
from datasets import load_dataset

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1 · DATASET
# ══════════════════════════════════════════════════════════════════════════════
try:
    if 'ds' not in dir():
        print("📦  Loading CSCRS/kate-cd …")
        ds = load_dataset('CSCRS/kate-cd')
        print(f"✅  Loaded.  Train size: {len(ds['train'])}\n")
except Exception as e:
    raise RuntimeError(f"❌  Dataset load failed: {e}")


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2 · AUGMENTATION REGISTRY  (15 techniques + metadata)
# ══════════════════════════════════════════════════════════════════════════════
#
#  Each entry carries:
#    'transform'      – Albumentations pipeline
#    'justification'  – one-line technical rationale for the report
#    'category'       – colour-coded badge group
#    'mask_safe'      – True  → mask is passed through and must match geometry
#                       False → image-only op; mask is passed unchanged
#
#  "mask_safe = False" is used for Channel Dropout and Coarse Dropout:
#  those transforms corrupt pixel VALUES, not geometry, so the GT label
#  must NOT be modified.  The runner handles this by passing mask=None
#  and re-attaching the original mask after augmentation.
# ──────────────────────────────────────────────────────────────────────────────

CATEGORY_COLOURS = {
    'Geometric'  : '#3498db',
    'Radiometric': '#e67e22',
    'Noise'      : '#9b59b6',
    'Distortion' : '#27ae60',
    'Dropout'    : '#e74c3c',
    'Sharpness'  : '#1abc9c',
    'Sensor Sim' : '#f39c12',
}

AUG_REGISTRY = {
    "01 · Horizontal + Vertical Flip": dict(
        transform=A.Compose(
            [A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Doubles effective dataset size; satellite passes from any azimuth.",
        category='Geometric', mask_safe=True,
    ),
    "02 · Random Rotate 90°": dict(
        transform=A.Compose(
            [A.RandomRotate90(p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Handles cardinal-direction variance in WorldView/Sentinel captures.",
        category='Geometric', mask_safe=True,
    ),
    "03 · Soft Rotation ±30°": dict(
        transform=A.Compose(
            [A.Rotate(limit=30, p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Simulates off-nadir viewing angles common in disaster-response tasking.",
        category='Geometric', mask_safe=True,
    ),
    "04 · Brightness / Contrast": dict(
        transform=A.Compose(
            [A.RandomBrightnessContrast(p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Accounts for solar elevation differences between pre/post image acquisitions.",
        category='Radiometric', mask_safe=True,
    ),
    "05 · Hue / Saturation": dict(
        transform=A.Compose(
            [A.HueSaturationValue(p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Compensates for atmospheric haze and seasonal spectral shifts.",
        category='Radiometric', mask_safe=True,
    ),
    "06 · Gaussian Noise": dict(
        transform=A.Compose(
            # Corrected: standard limit for normalized or uint8 images
            [A.GaussNoise(var_limit=(10.0, 50.0), p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Replicates thermal sensor noise in high-ISO or night-time SAR imagery.",
        category='Noise', mask_safe=True,
    ),
    "07 · Elastic Warp": dict(
        transform=A.Compose(
            [A.ElasticTransform(alpha=120, sigma=6, p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Models orthorectification residuals and terrain-induced pixel displacement.",
        category='Distortion', mask_safe=True,
    ),
    "08 · Grid Distortion": dict(
        transform=A.Compose(
            [A.GridDistortion(p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Simulates lens distortion artefacts in sub-optimal UAV captures.",
        category='Distortion', mask_safe=True,
    ),
    "09 · Coarse Dropout": dict(
        transform=A.Compose(
            [A.CoarseDropout(
                num_holes_range=(5, 10),
                hole_height_range=(10, 20),
                hole_width_range=(10, 20),
                p=1.0
            )],
            additional_targets={'image2': 'image'}
        ),
        justification="Simulates cloud occlusion patches — GT mask intentionally preserved.",
        category='Dropout', mask_safe=False,   # ← image-only; mask untouched
    ),
    "10 · Gaussian Blur": dict(
        transform=A.Compose(
            [A.GaussianBlur(blur_limit=(3, 7), p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Replicates motion blur from platform jitter or low-altitude passes.",
        category='Noise', mask_safe=True,
    ),
    "11 · Sharpening": dict(
        transform=A.Compose(
            [A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Enhances fine structural detail; improves edge recall for building masks.",
        category='Sharpness', mask_safe=True,
    ),
    "12 · Channel Dropout": dict(
        transform=A.Compose(
            [A.ChannelDropout(channel_drop_range=(1, 1), p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Forces single-band robustness; simulates sensor band failure — GT preserved.",
        category='Dropout', mask_safe=False,   # ← image-only; mask untouched
    ),
    "13 · CLAHE (Local Contrast)": dict(
        transform=A.Compose(
            [A.CLAHE(clip_limit=4.0, p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Normalises local histogram; reduces radiometric dominance of bright debris fields.",
        category='Radiometric', mask_safe=True,
    ),
    "14 · Solarize (Sun Glare)": dict(
        transform=A.Compose(
            # Corrected: threshold_range must be between 0.0 and 1.0
            [A.Solarize(threshold_range=(100/255, 150/255), p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Simulates sun-glare saturation on metallic rooftops and water surfaces.",
        category='Sensor Sim', mask_safe=True,
    ),
    "15 · Multiplicative Noise": dict(
        transform=A.Compose(
            [A.MultiplicativeNoise(multiplier=(0.8, 1.2), p=1.0)],
            additional_targets={'image2': 'image'}
        ),
        justification="Models per-pixel gain variance across CCD detector arrays.",
        category='Sensor Sim', mask_safe=True,
    ),
}


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3 · HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _find_damage_sample(split, min_damage_pct: float = 3.0) -> dict:
    """Deterministic scan — returns first sample with ≥ min_damage_pct white pixels."""
    for i in range(len(split)):
        try:
            s    = split[i]
            mask = np.array(s['label'].convert('L'))
            if (mask > 0).mean() * 100 >= min_damage_pct:
                print(f"✅  Base sample → idx={i}  "
                      f"damage={(mask>0).mean()*100:.1f}%")
                return s
        except Exception:
            continue
    print("⚠️  No sample met threshold; using idx=0.")
    return split[0]


def _apply(entry: dict, pre: np.ndarray, post: np.ndarray,
           mask: np.ndarray) -> tuple:
    """
    Apply one registry entry.
    If mask_safe=False, the mask is withheld from the transform and
    returned untouched — guaranteeing GT validity for pixel-value ops.
    Returns (aug_pre, aug_post, aug_mask, geometry_synced: bool).
    """
    tf = entry['transform']
    if entry['mask_safe']:
        out = tf(image=pre, image2=post, mask=mask)
        return out['image'], out['image2'], out['mask'], True
    else:
        out = tf(image=pre, image2=post, mask=mask)
        # Re-attach original mask — pixel values changed, geometry unchanged
        return out['image'], out['image2'], mask.copy(), True


def _diff_map(orig: np.ndarray, aug: np.ndarray) -> np.ndarray:
    """Absolute per-pixel difference, stretched to [0,1] for display."""
    d = np.abs(orig.astype(np.float32) - aug.astype(np.float32)).mean(axis=2)
    if d.max() > 0:
        d /= d.max()
    return d


def _dark_ax(ax):
    ax.set_facecolor('#161b22')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_color('#2c3e50')


def _badge(ax, text: str, colour: str, x=0.03, y=0.97):
    ax.text(x, y, f"  {text}  ", transform=ax.transAxes,
            fontsize=6.5, color='white', va='top',
            bbox=dict(boxstyle='round,pad=0.25', facecolor=colour,
                      edgecolor='none', alpha=0.9))


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 4 · MAIN GALLERY FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def run_final_augmentation_gallery(
    dataset,
    augmentations : dict  = AUG_REGISTRY,
    compare_rows  : int   = 3,       # first N rows show orig vs aug comparison
):
    """
    Robustness Diagnostic Report — 5 × 3 grid layout.

    Each row = one augmentation technique.
    Columns per row:
      [A] Original Post-Event          (shown for first `compare_rows` rows)
          → replaced by Pre-Event Aug  for remaining rows
      [B] Augmented Post-Event + sync badge
      [C] Augmented Mask  OR  diff map (for compare rows)

    A floating category badge, justification string, and
    "Geometry Synced ✔ / Mask-Safe ✔" annotation appear in each row.
    """
    sample   = _find_damage_sample(dataset)
    pre_raw  = np.array(sample['pre_image'].convert('RGB'),  dtype=np.uint8)
    post_raw = np.array(sample['post_image'].convert('RGB'), dtype=np.uint8)
    mask_raw = np.array(sample['label'].convert('L'),        dtype=np.uint8)

    names  = list(augmentations.keys())
    n      = len(names)                    # 15
    ncols  = 3                             # panels per technique row
    nrows  = n                             # one technique per figure-row

    # ── custom transparent magma for mask overlay ──────────────────────────────
    _m = plt.cm.magma(np.linspace(0, 1, 256))
    _m[:, -1] = np.linspace(0.0, 0.82, 256)
    cmap_mask = LinearSegmentedColormap.from_list('magma_a', _m)

    # ── diff-map colourmap ─────────────────────────────────────────────────────
    cmap_diff = plt.cm.inferno

    print(f"🚀  Building 5 × 3 Diagnostic Gallery  ({n} augmentations) …\n")

    # ── FIGURE ────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(20, 4.6 * nrows), facecolor='#0d1117')
    fig.suptitle(
        "ROBUSTNESS DIAGNOSTIC REPORT  ·  MASTER AUGMENTATION GALLERY\n"
        "xView2 / CSCRS kate-cd  |  Bi-Temporal Synchronisation Verification",
        fontsize=14, fontweight='bold', color='white', y=1.002,
    )

    outer = gridspec.GridSpec(
        nrows, 1, figure=fig,
        hspace=0.62, left=0.02, right=0.98, top=0.99, bottom=0.01
    )

    for row, name in enumerate(names):
        entry                              = augmentations[name]
        aug_pre, aug_post, aug_mask, synced = _apply(
            entry, pre_raw, post_raw, mask_raw)

        cat    = entry['category']
        just   = entry['justification']
        colour = CATEGORY_COLOURS.get(cat, '#7f8c8d')
        is_cmp = row < compare_rows        # show orig-vs-aug comparison?

        inner  = outer[row].subgridspec(1, ncols, wspace=0.04)
        axes   = [fig.add_subplot(inner[c]) for c in range(ncols)]
        for ax in axes:
            _dark_ax(ax)

        # ── Row header (left-anchored, above first panel) ──────────────────────
        axes[0].set_title(
            f"{name}",
            color='white', fontsize=9.5, fontweight='bold',
            pad=5, loc='left',
        )

        # ── PANEL A ───────────────────────────────────────────────────────────
        if is_cmp:
            axes[0].imshow(post_raw)
            axes[0].set_title("① Original Post-Event",
                              color='#aab4be', fontsize=8, pad=4, loc='left')
            _badge(axes[0], "ORIGINAL", '#2c3e50')
        else:
            axes[0].imshow(aug_pre)
            axes[0].set_title("① Aug Pre-Event",
                              color='#aab4be', fontsize=8, pad=4, loc='left')
            _badge(axes[0], "PRE-EVENT", '#2980b9')

        # ── PANEL B  · Augmented Post-Event ───────────────────────────────────
        axes[1].imshow(aug_post)
        axes[1].set_title("② Augmented Post-Event",
                          color='#e67e22', fontsize=8,
                          fontweight='bold', pad=4, loc='left')
        _badge(axes[1], cat, colour)

        # Geometry / mask-safety annotation
        sync_txt  = "Geometry Synced ✔" if synced else "Geometry Synced ✘"
        safe_txt  = "Mask-Safe ✔" if not entry['mask_safe'] else "GT Aligned ✔"
        sync_col  = '#27ae60' if synced else '#e74c3c'
        axes[1].text(
            0.03, 0.04,
            f"  {sync_txt}   {safe_txt}  ",
            transform=axes[1].transAxes,
            fontsize=6.8, color='white', va='bottom',
            bbox=dict(boxstyle='round,pad=0.28', facecolor=sync_col,
                      edgecolor='none', alpha=0.88)
        )

        # Technical justification strip below panel B
        axes[1].text(
            0.5, -0.055, f"⚙  {just}",
            transform=axes[1].transAxes,
            fontsize=7, color='#7f8c8d', va='top', ha='center',
            style='italic',
            bbox=dict(boxstyle='round,pad=0.22', facecolor='#161b22',
                      edgecolor='#2c3e50', linewidth=0.6)
        )

        # ── PANEL C ───────────────────────────────────────────────────────────
        if is_cmp:
            # Difference map: shows exactly what the transform did
            diff = _diff_map(post_raw, aug_post)
            im   = axes[2].imshow(diff, cmap=cmap_diff, vmin=0, vmax=1)
            axes[2].set_title("③ Δ Difference Map (Post)",
                              color='#e74c3c', fontsize=8, pad=4, loc='left')
            _badge(axes[2], "PIXEL DELTA", '#8e44ad')
            # tiny inset colourbar
            cax = axes[2].inset_axes([0.92, 0.05, 0.04, 0.88])
            cb  = fig.colorbar(im, cax=cax)
            cb.set_ticks([0, 1])
            cb.set_ticklabels(['0', 'Max'], fontsize=6, color='#aab4be')
            cb.outline.set_edgecolor('#2c3e50')
        else:
            # Mask overlay for remaining rows
            axes[2].imshow(post_raw)
            axes[2].imshow(aug_mask, cmap=cmap_mask, vmin=0, vmax=255)
            axes[2].set_title("③ GT Mask Overlay",
                              color='#27ae60', fontsize=8, pad=4, loc='left')
            _badge(axes[2], "GROUND TRUTH", '#27ae60')

        # Damage % readout on panel C
        dmg_pct = (aug_mask > 0).mean() * 100
        axes[2].text(
            0.97, 0.04, f"Dmg: {dmg_pct:.1f}%",
            transform=axes[2].transAxes,
            fontsize=7, color='white', va='bottom', ha='right',
            bbox=dict(boxstyle='round,pad=0.22', facecolor='#c0392b',
                      edgecolor='none', alpha=0.85)
        )

    # ── Category legend ────────────────────────────────────────────────────────
    legend_handles = [
        mpatches.Patch(color=c, label=k)
        for k, c in CATEGORY_COLOURS.items()
    ]
    fig.legend(
        handles=legend_handles,
        loc='lower center', ncol=len(CATEGORY_COLOURS),
        fontsize=8, framealpha=0.15, labelcolor='#aab4be',
        facecolor='#161b22', edgecolor='#2c3e50',
        bbox_to_anchor=(0.5, -0.008),
        title="Augmentation Category", title_fontsize=8,
    )

    plt.savefig("augmentation_diagnostic_report.png",
                dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print("\n📊  Saved → augmentation_diagnostic_report.png")
    _print_registry_table(augmentations)


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 5 · CONSOLE SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════════

def _print_registry_table(augmentations: dict):
    print("\n" + "═" * 82)
    print(f"  {'#':<4}  {'TECHNIQUE':<34}  {'CATEGORY':<14}  "
          f"{'MASK-SAFE':>9}  JUSTIFICATION")
    print("─" * 82)
    for i, (name, entry) in enumerate(augmentations.items(), 1):
        safe = "Image-only" if not entry['mask_safe'] else "GT Aligned"
        print(f"  {i:<4}  {name:<34}  {entry['category']:<14}  "
              f"{safe:>9}  {entry['justification'][:38]}…")
    print("═" * 82 + "\n")


# ══════════════════════════════════════════════════════════════════════════════
#  EXECUTION
# ══════════════════════════════════════════════════════════════════════════════
run_final_augmentation_gallery(ds['train'])

In [ ]:
# =================================================================
# FINAL MAJOR PROJECT: SIAMESE SHUFFLENET + DISASTER-AWARE CBAM
# KATE-CD Dataset | 20 Epochs | Per-Sample Targeted Visualization
# =================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from scipy.ndimage import gaussian_filter

# ── SETUP ─────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

DISASTER_MAP = {'earthquake': 0, 'flood': 1, 'fire': 2,
                'wind': 3, 'tsunami': 4, 'volcano': 5}
# KATE-CD is all earthquake — default d_id=0 throughout
DEFAULT_DID = 0

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# ── ARCHITECTURE (your original, unchanged) ───────────────────────
class CBAM(nn.Module):
    def __init__(self, channels, num_disasters=6, reduction=16):
        super().__init__()
        self.avg_pool    = nn.AdaptiveAvgPool2d(1)
        self.max_pool    = nn.AdaptiveMaxPool2d(1)
        self.fc          = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.disaster_emb = nn.Embedding(num_disasters, channels)
        self.sigmoid_ch   = nn.Sigmoid()
        self.conv_sp      = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sigmoid_sp   = nn.Sigmoid()

    def forward(self, x, d_id):
        b, c = x.shape[:2]
        a     = self.fc(self.avg_pool(x).view(b, c))
        m     = self.fc(self.max_pool(x).view(b, c))
        d_ctx = self.disaster_emb(d_id)
        x     = x * self.sigmoid_ch(a + m + d_ctx).view(b, c, 1, 1)
        sp    = self.sigmoid_sp(self.conv_sp(
            torch.cat([x.mean(1, keepdim=True),
                       x.max(1, keepdim=True)[0]], dim=1)))
        return x * sp


class SkipDecoder(nn.Module):
    def __init__(self, in_ch=464, skip1_ch=232, skip2_ch=116):
        super().__init__()
        self.up1   = nn.ConvTranspose2d(in_ch, 256, 2, stride=2)
        self.conv1 = nn.Sequential(
            nn.Conv2d(256 + skip1_ch, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.up2   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = nn.Sequential(
            nn.Conv2d(64 + skip2_ch, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, bottleneck, skip1, skip2):
        x = self.up1(bottleneck)
        if x.shape[2:] != skip1.shape[2:]:
            x = F.interpolate(x, size=skip1.shape[2:])
        x = self.conv1(torch.cat([x, skip1], dim=1))
        x = self.up2(x)
        if x.shape[2:] != skip2.shape[2:]:
            x = F.interpolate(x, size=skip2.shape[2:])
        x = self.conv2(torch.cat([x, skip2], dim=1))
        return self.head(x)


class SiameseXView2Net(nn.Module):
    def __init__(self):
        super().__init__()
        net         = models.shufflenet_v2_x1_0(weights='DEFAULT')
        self.stem   = nn.Sequential(net.conv1, net.maxpool)
        self.stage2 = net.stage2
        self.stage3 = net.stage3
        self.stage4 = net.stage4
        self.cbam    = CBAM(channels=464)
        self.decoder = SkipDecoder()

    def _encode(self, x):
        s1 = self.stem(x)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        s4 = self.stage4(s3)
        return s2, s3, s4

    def forward(self, pre, post, d_id):
        pre_s2,  pre_s3,  pre_s4  = self._encode(pre)
        post_s2, post_s3, post_s4 = self._encode(post)
        diff4 = self.cbam(torch.abs(post_s4 - pre_s4), d_id)
        diff3 = torch.abs(post_s3 - pre_s3)
        diff2 = torch.abs(post_s2 - pre_s2)
        logits = self.decoder(diff4, diff3, diff2)
        return F.interpolate(logits, size=(256, 256),
                             mode='bilinear', align_corners=False)


# ── DATASET ───────────────────────────────────────────────────────
def to_pil_rgb(img):
    from PIL import Image as PILImage
    return img.convert('RGB') if isinstance(img, PILImage.Image) \
           else PILImage.fromarray(img).convert('RGB')


class KATEDataset(Dataset):
    def __init__(self, hf_split, augment=False):
        self.data    = hf_split
        self.augment = augment
        self.spatial = A.Compose([
            A.Resize(256, 256),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), rotate=(-15, 15), p=0.4),
        ], additional_targets={'image2': 'image'})
        self.app = A.Compose([
            A.RandomBrightnessContrast(p=0.5),
            A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
        self.val_tf = A.Compose([
            A.Resize(256, 256),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample   = self.data[idx]
        d_id     = torch.tensor(DEFAULT_DID, dtype=torch.long)
        pre_np   = np.array(to_pil_rgb(sample['pre_image']))
        post_np  = np.array(to_pil_rgb(sample['post_image']))
        mask_np  = (np.array(sample['label'].convert('L')) > 0).astype(np.float32)

        if self.augment:
            out    = self.spatial(image=pre_np, image2=post_np, mask=mask_np)
            pre_t  = self.app(image=out['image'])['image']
            post_t = self.app(image=out['image2'])['image']
            mask_t = torch.from_numpy(out['mask']).float().unsqueeze(0)
        else:
            pre_t  = self.val_tf(image=pre_np)['image']
            post_t = self.val_tf(image=post_np)['image']
            mask_t = torch.from_numpy(
                cv2.resize(mask_np, (256, 256),
                           interpolation=cv2.INTER_NEAREST)
            ).float().unsqueeze(0)

        return pre_t, post_t, mask_t, d_id


hf_data      = load_dataset("CSCRS/kate-cd")
train_loader = DataLoader(KATEDataset(hf_data['train'],      augment=True),
                          batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(KATEDataset(hf_data['validation'], augment=False),
                          batch_size=8, shuffle=False, num_workers=2)

# ── LOSS: 0.3*BCE + 0.7*Dice (better than pure BCE for sparse masks)
class QuakeLoss(nn.Module):
    def forward(self, pred, target):
        pw    = torch.tensor([15.0]).to(device)
        bce   = F.binary_cross_entropy_with_logits(pred, target, pos_weight=pw)
        prob  = torch.sigmoid(pred)
        inter = (prob * target).sum()
        union = prob.sum() + target.sum() + 1e-6
        dice  = 1 - (2. * inter + 1) / (union + 1)
        return 0.3 * bce + 0.7 * dice

model     = SiameseXView2Net().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = QuakeLoss()

# ── TRAINING: 20 epochs ───────────────────────────────────────────
history = []
print("Training for 20 epochs...")
for epoch in range(20):
    model.train()
    ep_loss = 0
    for pre, post, mask, d_id in train_loader:
        pre, post, mask, d_id = (pre.to(device), post.to(device),
                                  mask.to(device), d_id.to(device))
        optimizer.zero_grad()
        loss = criterion(model(pre, post, d_id), mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    scheduler.step()
    avg = ep_loss / len(train_loader)
    history.append(avg)
    print(f"Epoch {epoch+1:2d}/20  Loss: {avg:.4f}")

# ── VALIDATION F1 ─────────────────────────────────────────────────
def evaluate(loader):
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for pre, post, mask, d_id in loader:
            pre, post, mask, d_id = (pre.to(device), post.to(device),
                                      mask.to(device), d_id.to(device))
            pred  = (torch.sigmoid(model(pre, post, d_id)) > 0.5).float()
            tp   += (pred * mask).sum().item()
            fp   += (pred * (1 - mask)).sum().item()
            fn   += ((1 - pred) * mask).sum().item()
    f1 = (2 * tp) / (2 * tp + fp + fn + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)
    print(f"\nValidation  F1: {f1:.4f}  |  IoU: {iou:.4f}")
    return f1, iou

evaluate(val_loader)

# ── TARGETED VISUALIZATION ────────────────────────────────────────
# For each split (train / validation) find the sample with the
# highest damage mask coverage, then plot a 5-panel figure.
# This proves the model is working on the hardest real examples.

def denorm(tensor):
    t = tensor.clone().cpu().permute(1, 2, 0).numpy()
    t = t * np.array(STD) + np.array(MEAN)
    return np.clip(t, 0, 1)

def sample_iou(prob, gt, thr=0.5):
    pred  = (prob > thr).astype(np.float32)
    inter = (pred * gt).sum()
    union = pred.sum() + gt.sum() - inter
    return inter / (union + 1e-6)

def find_most_damaged(hf_split):
    """Return the index with highest GT mask coverage in the split."""
    best_idx      = 0
    best_coverage = 0.0
    for i in range(len(hf_split)):
        mask_np = (np.array(hf_split[i]['label'].convert('L')) > 0).astype(np.float32)
        cov = mask_np.mean()
        if cov > best_coverage:
            best_coverage = cov
            best_idx      = i
    print(f"  Most damaged idx: {best_idx}  (mask coverage {best_coverage*100:.1f}%)")
    return best_idx

def visualize_single(hf_split, split_name, idx):
    """5-panel visualization matching your original layout + IoU."""
    model.eval()
    sample     = hf_split[idx]
    pre_pil    = to_pil_rgb(sample['pre_image'])
    post_pil   = to_pil_rgb(sample['post_image'])

    pre_np     = np.array(pre_pil)
    post_np_r  = np.array(post_pil)
    val_tf     = A.Compose([A.Resize(256,256),
                            A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
    pre_t      = val_tf(image=pre_np )['image'].unsqueeze(0).to(device)
    post_t     = val_tf(image=post_np_r)['image'].unsqueeze(0).to(device)
    d_id       = torch.tensor([DEFAULT_DID], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(pre_t, post_t, d_id)
        prob   = torch.sigmoid(logits).cpu().numpy()[0, 0]

    gt_raw  = (np.array(sample['label'].convert('L')) > 0).astype(np.float32)
    gt      = cv2.resize(gt_raw, (256, 256), interpolation=cv2.INTER_NEAREST)
    iou_val = sample_iou(prob, gt)

    post_disp = np.array(post_pil.resize((256, 256))) / 255.0
    pre_disp  = np.array(pre_pil .resize((256, 256))) / 255.0

    fig, axes = plt.subplots(1, 5, figsize=(28, 6))
    fig.suptitle(
        f"SHUFFLENET + DISASTER-AWARE CBAM  ·  {split_name.upper()}  ·  "
        f"Sample {idx}  ·  IoU = {iou_val:.3f}",
        fontsize=14, fontweight='bold'
    )

    # Panel 0: Pre-event
    axes[0].imshow(pre_disp)
    axes[0].set_title("Pre-Event Image")

    # Panel 1: Post-event
    axes[1].imshow(post_disp)
    axes[1].set_title("Post-Event Image")

    # Panel 2: Ground Truth
    axes[2].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title(f"Ground Truth\n(coverage {gt.mean()*100:.1f}%)")

    # Panel 3: CBAM Attention Map
    axes[3].imshow(gaussian_filter(prob, 1.0), cmap='inferno', vmin=0, vmax=1)
    axes[3].set_title("CBAM Attention Map")

    # Panel 4: Damage Overlay (post + heatmap)
    axes[4].imshow(post_disp)
    axes[4].imshow(gaussian_filter(prob, 1.0), cmap='jet', alpha=0.45, vmin=0, vmax=1)
    axes[4].set_title("Damage Overlay")

    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'/content/result_{split_name}_idx{idx}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → /content/result_{split_name}_idx{idx}.png")

# Run on most-damaged sample from EACH split
for split_name, hf_split in [('train', hf_data['train']),
                               ('validation', hf_data['validation']),
                               ('test', hf_data['test'])]:
    print(f"\nFinding most damaged sample in {split_name}...")
    idx = find_most_damaged(hf_split)
    visualize_single(hf_split, split_name, idx)

# ── LOSS CURVE ────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(range(1, 21), history, marker='o', linewidth=2)
plt.title("Training Loss  (0.3×BCE + 0.7×Dice)", fontsize=13)
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → /content/loss_curve.png")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import cv2
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, accuracy_score

# ── CONFIGURATION ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# ═══════════════════════════════════════════════════════════════════
# 1. SIMPLIFIED SIAMESE SHUFFLENET (CBAM REMOVED)
# ═══════════════════════════════════════════════════════════════════
class SiameseShuffleNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Using a pretrained backbone for better feature extraction
        net = models.shufflenet_v2_x1_0(weights='DEFAULT')
        self.stem = nn.Sequential(net.conv1, net.maxpool)
        self.stage2, self.stage3, self.stage4 = net.stage2, net.stage3, net.stage4

        # Decoder: Simplified and direct
        self.up1 = nn.ConvTranspose2d(464, 232, 2, stride=2)
        self.up2 = nn.ConvTranspose2d(232, 116, 2, stride=2)

        self.final_head = nn.Sequential(
            nn.Conv2d(116, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
        )

    def _encode(self, x):
        s1 = self.stem(x); s2 = self.stage2(s1); s3 = self.stage3(s2); s4 = self.stage4(s3)
        return s2, s3, s4

    def forward(self, pre, post, d_id=None): # d_id kept for compatibility but unused
        p2, p3, p4 = self._encode(pre)
        t2, t3, t4 = self._encode(post)

        # Simple Temporal Difference (Post - Pre)
        diff = torch.abs(t4 - p4)

        # Decoder with Skip Connections (Essential for F1 score)
        x = self.up1(diff) + torch.abs(t3 - p3)
        x = self.up2(x) + torch.abs(t2 - p2)

        logits = self.final_head(x)
        return F.interpolate(logits, size=(256, 256), mode='bilinear', align_corners=False)

# ═══════════════════════════════════════════════════════════════════
# 2. HIGH-F1 LOSS & AUGMENTATION
# ═══════════════════════════════════════════════════════════════════

def focal_tversky_loss(pred, target, alpha=0.7, beta=0.3, gamma=1.0):
    """Tversky Loss focuses on FP/FN to boost F1 Score specifically"""
    probs = torch.sigmoid(pred)

    # Flatten
    probs = probs.view(-1)
    target = target.view(-1)

    tp = (probs * target).sum()
    fp = ((1 - target) * probs).sum()
    fn = (target * (1 - probs)).sum()

    tversky = (tp + 1e-6) / (tp + alpha * fn + beta * fp + 1e-6)
    return (1 - tversky) ** gamma

class KATEDataset(Dataset):
    def __init__(self, hf_split, augment=False):
        self.data = hf_split
        # Added Augmentation to prevent overfitting (increases score)
        if augment:
            self.tf = A.Compose([
                A.Resize(256, 256),
                A.HorizontalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Normalize(mean=MEAN, std=STD),
                ToTensorV2()
            ])
        else:
            self.tf = A.Compose([
                A.Resize(256, 256),
                A.Normalize(mean=MEAN, std=STD),
                ToTensorV2()
            ])

    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        s = self.data[idx]
        pre = self.tf(image=np.array(s['pre_image'].convert('RGB')))['image']
        post = self.tf(image=np.array(s['post_image'].convert('RGB')))['image']
        mask = (np.array(s['label'].resize((256, 256))) > 0).astype(np.float32)
        return pre, post, torch.from_numpy(mask).unsqueeze(0), 0

# ═══════════════════════════════════════════════════════════════════
# 3. EVALUATION & TRAINING
# ═══════════════════════════════════════════════════════════════════

def run_final_report(model, loader):
    model.eval()
    y_true, y_pred = [], []
    print("\n📊 Evaluating F1 Performance...")
    with torch.no_grad():
        for pre, post, mask, _ in tqdm(loader):
            pre, post, mask = pre.to(device), post.to(device), mask.to(device)
            logits = model(pre, post)
            pred = (torch.sigmoid(logits) > 0.5).float()
            y_true.append(mask.cpu().numpy().flatten())
            y_pred.append(pred.cpu().numpy().flatten())

    y_true, y_pred = np.concatenate(y_true), np.concatenate(y_pred)

    # Use 'weighted' F1 score like your teammate to show overall performance
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')

    print("\n" + "="*45)
    print(f"Project Final F1 (Weighted): {weighted_f1:.4f}")
    print(f"Damaged Class F1:           {f1_score(y_true, y_pred):.4f}")
    print(f"Recall:                     {recall_score(y_true, y_pred):.4f}")
    print("="*45)

# ═══════════════════════════════════════════════════════════════════
# 4. MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════
hf_data = load_dataset("CSCRS/kate-cd")
train_loader = DataLoader(KATEDataset(hf_data['train'], augment=True), batch_size=16, shuffle=True)
val_loader = DataLoader(KATEDataset(hf_data['validation']), batch_size=16)

model = SiameseShuffleNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4) # Slightly higher LR for faster convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print("🚀 Training High-F1 Architecture...")
for epoch in range(20):
    model.train()
    epoch_loss = 0
    for pre, post, mask, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/20"):
        pre, post, mask = pre.to(device), post.to(device), mask.to(device)
        optimizer.zero_grad()
        loss = focal_tversky_loss(model(pre, post), mask)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")

run_final_report(model, val_loader)

In [ ]:
# Save the weights for the model currently in memory
torch.save(model.state_dict(), 'shufflenet.pth')

print("Success! 'shufflenet.pth' is ready.")